In [1]:
import numpy as np
import pandas as pd
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, StandardScaler

In [2]:
df = pd.read_csv("../data/processed/cars_features.csv")
df.shape

(97675, 14)

In [3]:
TARGET = "price_log"
DROP_COLS = ["price", "price_log", "year", "brand_model"]

y = df[TARGET]
y_price = df["price"]
X = df.drop(columns=DROP_COLS)
X.columns.tolist()

['model',
 'transmission',
 'mileage',
 'fuelType',
 'tax',
 'mpg',
 'engineSize',
 'Make',
 'age',
 'mileage_per_year']

In [4]:
X_train, X_test, y_train, y_test, price_train, price_test = train_test_split(
    X, y, y_price, test_size=0.2, random_state=42
)
X_train.shape, X_test.shape

((78140, 10), (19535, 10))

In [5]:
NUMERIC = ["mileage", "tax", "mpg", "engineSize", "age", "mileage_per_year"]
LOW_CARD_CAT = ["transmission", "fuelType", "Make"]
HIGH_CARD_CAT = ["model"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), NUMERIC),
    ("onehot", OneHotEncoder(handle_unknown="ignore"), LOW_CARD_CAT),
    ("target_enc", TargetEncoder(random_state=42), HIGH_CARD_CAT),
])

In [6]:
X_train_proc = preprocessor.fit_transform(X_train, y_train)
X_test_proc = preprocessor.transform(X_test)

print("Train:", X_train_proc.shape)
print("Test: ", X_test_proc.shape)

Train: (78140, 25)
Test:  (19535, 25)


c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


In [7]:
feature_names = preprocessor.get_feature_names_out()
len(feature_names), feature_names[:10]

(25,
 array(['num__mileage', 'num__tax', 'num__mpg', 'num__engineSize',
        'num__age', 'num__mileage_per_year',
        'onehot__transmission_Automatic', 'onehot__transmission_Manual',
        'onehot__transmission_Other', 'onehot__transmission_Semi-Auto'],
       dtype=object))

# Mechya sugoi dane kore

In [8]:
OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

joblib.dump(preprocessor, f"{OUT_DIR}/preprocessor.joblib")
np.save(f"{OUT_DIR}/X_train.npy", X_train_proc)
np.save(f"{OUT_DIR}/X_test.npy", X_test_proc)
y_train.to_csv(f"{OUT_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{OUT_DIR}/y_test.csv", index=False)
price_test.to_csv(f"{OUT_DIR}/price_test.csv", index=False)
pd.Series(feature_names).to_csv(f"{OUT_DIR}/feature_names.csv", index=False, header=["feature"])

print("Saved preprocessor + train/test arrays to", OUT_DIR)

Saved preprocessor + train/test arrays to ../data/processed
